# Chapter 4: Advanced Analytics & AI Insights
In this notebook, we move beyond basic descriptive statistics to uncover hidden patterns and forecast future trends.

## 1. Merchant Fraud Ring Clustering (K-Means)
We use K-Means clustering to segment merchants into behavioral profiles based on their chargeback volume, average ticket size, and dispute ratio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load datasets
merchants = pd.read_csv('../data/processed/merchants_clean.csv')
cb_summary = pd.read_csv('../data/processed/chargeback_transaction_summary.csv')
txns = pd.read_csv('../data/processed/upi_transactions_clean.csv')

# Feature Engineering for Clustering
# 1. Total volume per merchant
merch_vol = txns.groupby('merchant_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_txn_volume'})
# 2. Total chargeback amount per merchant
cb_txns = pd.merge(cb_summary, txns[['txn_id', 'merchant_id']], on='txn_id', how='left')
merch_cb = cb_txns.groupby('merchant_id')['chargeback_amount'].sum().reset_index()

# Combine features
df_cluster = pd.merge(merch_vol, merch_cb, on='merchant_id', how='left').fillna(0)
df_cluster['cb_ratio'] = (df_cluster['chargeback_amount'] / df_cluster['total_txn_volume']).fillna(0)
df_cluster = df_cluster.replace([np.inf, -np.inf], 0)

# Scale data
features = df_cluster[['total_txn_volume', 'chargeback_amount', 'cb_ratio']]
scaler = StandardScaler()
features = features.fillna(0)
scaled_features = scaler.fit_transform(features)

# Apply K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(scaled_features)

# Map clusters to profiles
cluster_names = {0: 'Low Risk (Standard)', 1: 'High Volume/Low Dispute', 2: 'High Risk (Fraud Ring / Suspicious)'}
df_cluster['profile'] = df_cluster['cluster'].map(cluster_names)

print(df_cluster['profile'].value_counts())

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_cluster, x='total_txn_volume', y='cb_ratio', hue='profile', palette='bright')
plt.title('Merchant Clustering: Transaction Volume vs Chargeback Ratio')
plt.show()


## 2. Dispute Volume Forecasting (ARIMA / Simple Smoothing)
We will aggregate daily chargebacks and use Simple Exponential Smoothing to forecast the next 7 days of dispute volume.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

cb_data = pd.read_csv('../data/processed/chargebacks_clean.csv')
cb_data['transaction_timestamp'] = pd.to_datetime(cb_data['transaction_timestamp'])
daily_cb = cb_data.groupby(cb_data['transaction_timestamp'].dt.date).size().reset_index(name='count')
daily_cb.set_index('transaction_timestamp', inplace=True)
daily_cb.index = pd.to_datetime(daily_cb.index)
daily_cb = daily_cb.asfreq('D').fillna(0)

# Fit model
model = SimpleExpSmoothing(daily_cb['count'], initialization_method='estimated').fit()
forecast = model.forecast(7)

plt.figure(figsize=(12, 5))
plt.plot(daily_cb.index, daily_cb['count'], label='Actual Daily Chargebacks')
plt.plot(forecast.index, forecast, label='7-Day Forecast', linestyle='--', color='red')
plt.title('Daily Chargeback Volume & Forecast')
plt.legend()
plt.show()


## 3. Dispute Insights & Cohort Analysis
Here we visualize the distribution of dispute reasons and intake channels to identify systemic vulnerabilities.

In [ ]:
# Chargeback Reason Code Distribution
plt.figure(figsize=(10, 6))
reason_counts = cb_data['reason_code'].value_counts()
sns.barplot(x=reason_counts.values, y=reason_counts.index, palette='rocket')
plt.title('Dispute Distribution by Reason Code')
plt.xlabel('Number of Chargebacks')
plt.ylabel('Reason Code')
plt.show()

In [ ]:
# Intake Channel Distribution
plt.figure(figsize=(8, 8))
channel_counts = cb_data['channel'].value_counts()
plt.pie(channel_counts.values, labels=channel_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Chargeback Volume by Intake Channel')
plt.show()

In [ ]:
# Transaction Volume Time-of-Day Heatmap
txns['timestamp'] = pd.to_datetime(txns['timestamp'])
txns['hour_of_day'] = txns['timestamp'].dt.hour
txns['day_of_week'] = txns['timestamp'].dt.day_name()
heatmap_data = pd.pivot_table(txns, values='amount', index='day_of_week', columns='hour_of_day', aggfunc='count', fill_value=0)

# Reorder days
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.5)
plt.title('Transaction Frequency Heatmap (Hour vs Day of Week)')
plt.xlabel('Hour of Day (24H)')
plt.ylabel('Day of Week')
plt.show()